# Step 4 prep: semi-synthetic case2 (`case2s`) by street swap

Adds look-alike-street negatives to every split. X = a place, Y = a place on a different but similar-looking street
in the same ZIP; the new pair is X's Overture address vs **X's house number + Y's Foursquare street text**, label 0.
Swaps **replace** the same number of easy negatives per split, so every split stays 1:9. Reads the final pairs from
`02_processing_overture_data.ipynb` (via `load_pairs`); run 01 → 02 → this notebook.

Spec agreed 2026-10-27 (full tables in `logs/progress-history.md`, 2026-10-27).

**How this notebook is laid out**
- **Pipeline (§0–§13):** each section = one markdown cell (what it does, input → output variable, which diagnostic
  reads it) + one code cell. The first line of each code cell says what that cell is for.
- **Acceptance checks:** asserts every swap row must pass (a function + a test cell on hand-made rows).
- **Diagnostics (D0–D5):** one-off inspections, one markdown + one code cell each. Results are written into the
  markdown (question → result → query), not left as code output.

| § | Stage | Output variable | Read by |
|---|---|---|---|
| 0 | Setup | | |
| — | Load the final pairs | `pairs` | D0 |
| 1 | Load places | `places` | |
| 2 | Clean house numbers only | `places` | |
| 3 | Overture and Foursquare streets agree | `places` | D5 |
| 4 | Unique streets per (split, ZIP) | `streets` | |
| 5 | Street × street pairs within a ZIP | `street_pairs` | |
| 6 | Drop same street, written differently | `street_pairs` | D5 |
| 7 | Drop directional-only pairs | `street_pairs` | D5 |
| 8 | Look-alike: JW > 0.8 | `street_pairs` | D1 |
| 9 | `swap_type` | `street_pairs` | D1 |
| 10 | Back to places (X, Y) | `swaps` | |
| 11 | Build the pairs | `swaps` | D2, D3, D4, acceptance |
| 12 | Counts → decide amounts (D6) | | |
| 13 | Sample, replace easy negatives, write | new pair files | acceptance (1:9) |

**Still open:** D4 (this notebook vs a section in 02: this file assumes the new notebook), D5 (source = final pairs'
positives vs the pre-dedupe table: §1 assumes final pairs), D6 (per-X cap, train share, eval size: after §12).

## 0. Setup

In [1]:
# §0: imports and constants
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import jellyfish
from postal.expand import expand_address

from helpers.pairs import load_pairs, add_number_street
from helpers.check_model_performance import find_random_sample_given_mask

OVERTURE_RELEASE = "2026-09-23.1"   # must match 01 / 02
SEED = 0

## Load the final pairs
**Does:** all three splits in one frame, label column re-attached. Nothing filtered yet.  
**Output:** `pairs` · **Read by:** D0

In [2]:
# Load: all splits -> `pairs` (also the lookup for the "not already in any split" acceptance check)
splits = []
for split in ["train", "val", "test"]:
    df, label = load_pairs("overture", split)
    splits.append(df.assign(label=label))
pairs = pd.concat(splits, ignore_index=True)

print(pairs.shape)
pairs.groupby(["split", "neg_type"], observed=True).size().unstack()

(288277, 9)


neg_type,case0,case1,case2,case3
split,,,,
train,20259,67174,172,114963
val,4187,13781,34,23863
test,4386,14448,15,24995


## 1. Load places
**Does:** Keep positives (`case0`) → `add_number_street`. One row = one place (X or Y candidate).  
**Input → output:** `pairs` → `places`

In [ ]:
# §1: Load places


## 2. Clean house numbers only
**Does:** Keep rows where `number_a` and `number_b` are plain digits (no `16-18`, `20A`, missing).  
**Input → output:** `places` → `places`

In [ ]:
# §2: Clean house numbers only

## 3. Overture and Foursquare streets agree
**Does:** Keep places whose `street_a` and `street_b` agree. Stops a Y whose Foursquare text names X's street from producing a true match labelled 0. Keep the dropped rows in a variable for D5.  
**Input → output:** `places` → `places` · **Read by:** D5

In [ ]:
# §3: Overture and Foursquare streets agree

## 4. Unique streets
**Does:** Unique `(split, zip_a, street_a)`.  
**Input → output:** `places` → `streets`

In [ ]:
# §4: Unique streets

## 5. Street pairs
**Does:** Self-join `streets` on `(split, zip_a)`; keep pairs with different names.  
**Input → output:** `streets` → `street_pairs`

In [ ]:
# §5: Street pairs

## 6. Drop same street, written differently
**Does:** `expand_address` sets intersect → drop. Expand each unique street once (dict), not per pair. Keep the dropped rows for D5.  
**Input → output:** `street_pairs` → `street_pairs` · **Read by:** D5

In [ ]:
# §6: Drop same street, written differently

## 7. Drop directional-only
**Does:** Names equal after removing `N S E W NE NW SE SW NORTH SOUTH EAST WEST` → drop (logged for later: coordinate clustering per street). Keep the dropped rows for D5.  
**Input → output:** `street_pairs` → `street_pairs` · **Read by:** D5

In [ ]:
# §7: Drop directional-only

## 8. Look-alike
**Does:** Keep JW > 0.8; keep `jw` as a column.  
**Input → output:** `street_pairs` → `street_pairs` · **Read by:** D1

In [ ]:
# §8: Look-alike

## 9. `swap_type`
**Does:** `suffix_only` if all tokens but the last are equal, else `name_differs`.  
**Input → output:** `street_pairs` → `street_pairs` · **Read by:** D1

In [ ]:
# §9: swap_type

## 10. Back to places
**Does:** X = a place on street 1, Y = a place on street 2 (both directions of each street pair).  
**Input → output:** `street_pairs` + `places` → `swaps`

In [ ]:
# §10: Back to places

## 11. Build the pairs
**Does:** `address_a` = X's A · `address_b` = X's number + " " + Y's `street_b` · `zip_a` = X's · `zip_b` = Y's · `id_a` = X · `id_b` = Y · `split` · label 0 · `neg_type = "case2s"` · keep `jw`, `swap_type`, and add `street_a` (X's) and `street_b` (Y's, the street in the new B).  
**Input → output:** `swaps` → `swaps` (shared schema + `jw`, `swap_type`, `street_a`, `street_b`) · **Read by:** D2, D3, D4, acceptance

In [ ]:
# §11: Build the pairs

## Acceptance checks
Every `case2s` row must pass these. Run on `swaps` after §11; the 1:9 check runs after §13.

| Check | Type |
|---|---|
| `zip_a == zip_b`; same split; `id_a != id_b` | flagged |
| number in the new B == `number_a` | flagged |
| X's `street_a` vs the new B's street: no libpostal overlap | flagged |
| `(address_a, address_b)` unique among the swap rows | flagged |
| B matches `^\d+ \S`, no double spaces | flagged |
| per split, positives and negatives after §13 == counts in `pairs` (replacement, not added on top) | flagged |

**Known limits (2026-10-27, David's call):**
- #1 number and #3 street read the `number_*` / `street_*` columns made in §1 (the breakdown of the original
  address), not a re-parse of the built `address_b`. A §11 bug that builds the string wrong but keeps the columns
  right would pass.
- #3 libpostal overlap misses the same street when B carries extra text: `FARMINGTON AVE` → `{farmington avenue}`
  vs `Farmington Ave, Ste 2` → `{farmington avenue suite 2, …}`, no overlap. Fixing it needs the `road` part of B
  (e.g. `parse_address`); not done (time). Same limit applies to §3.

In [18]:
# Acceptance: function that takes the full pair set (+ `pairs`) and returns a list of problems
def flag_issues(df, pairs):
    issues = []  # [Error Message, Sample of Examples]
    checks = {}  # message -> mask of the rows that VIOLATE the check

    # 6. swaps REPLACE easy negatives: per split, positives and negatives after §13 must equal the counts in `pairs`
    #    (the set before §13). Exact counts, not a share: 400 val swaps added on top only move the share 0.900 -> 0.901.
    #    Runs on the full set, before the neg_type filter.
    before = pairs.groupby(["split", "label"], observed=True).size()
    after = df.groupby(["split", "label"], observed=True).size()
    for (split, label), n_before in before.items():
        n_after = after.get((split, label), 0)
        if n_after != n_before:
            kind = "positives" if label == 1 else "negatives"
            issues.append([f"#6 {split}: {n_after} {kind}, expected {n_before} (as in `pairs`)", df[df["split"] == split].head()])

    df = df[df["neg_type"].isin(["case2", "case2s"])].copy()

    # 1. ids / zip / number
    checks["#1 id_a equals id_b"] = df["id_a"] == df["id_b"]
    checks["#1 zip_a doesn't match zip_b"] = df["zip_a"] != df["zip_b"]
    has_number = df["number_a"].notna()  # only judge the number when A has one
    checks["#1 number_a doesn't match number_b"] = has_number & (df["number_a"] != df["number_b"])

    # 2. split: id -> split lookup from both id columns of `pairs`
    lookup = (
        pd.concat([
            pairs[["id_a", "split"]].rename(columns={"id_a": "id"}),
            pairs[["id_b", "split"]].rename(columns={"id_b": "id"}),
        ])
        .drop_duplicates()
        .set_index("id")["split"]
    )
    df["i_split_a"] = df["id_a"].map(lookup)
    df["i_split_b"] = df["id_b"].map(lookup)
    checks["#2 id_a or id_b is not in the row's split"] = ~((df["i_split_a"] == df["i_split_b"]) & (df["i_split_a"] == df["split"]))

    # 3. libpostal overlap
    unique_names = set(df["street_a"].unique().tolist() + df["street_b"].unique().tolist())
    encoding_libpostal = {name: set(expand_address(name, languages=["en"])) for name in unique_names}

    df["libpostal_overlap_a"] = df["street_a"].map(encoding_libpostal)
    df["libpostal_overlap_b"] = df["street_b"].map(encoding_libpostal)
    checks["#3 street_a and street_b share a libpostal expansion"] = pd.Series(
        [len(a & b) > 0 for a, b in zip(df["libpostal_overlap_a"], df["libpostal_overlap_b"])],
        index=df.index,
    )

    # 4. (address_a, address_b) unique
    checks["#4 (address_a, address_b) is duplicated"] = (df["address_a"] + "|" + df["address_b"]).duplicated(keep=False)

    # 5. extra whitespace
    checks["#5 address_b doesn't match ^\\d+ \\S"] = has_number & ~df["address_b"].str.contains(r"^\d+ \S", na=False)

    for message, bad in checks.items():
        if bad.any():
            issues.append([f"{message}. Rows mismatch: {bad.sum()}", df[bad].head()])

    return issues


In [ ]:
# Acceptance tests: a 1:9 frame (per split: 1 positive, 8 easy negatives, 1 swap); each bad edit must be flagged by its own check
splits = ["train", "val", "test"]
numbers = {"train": "12", "val": "14", "test": "16"}

def make_row(split, id_a, id_b, label, neg_type):
    n = numbers[split]
    return {
        "id_a": id_a, "id_b": id_b,
        "address_a": f"{n} Main Street", "address_b": f"{n} Maple Avenue",
        "street_a": "Main Street", "street_b": "Maple Avenue",
        "zip_a": "06901", "zip_b": "06901",
        "number_a": n, "number_b": n,
        "split": split, "label": label, "neg_type": neg_type,
    }

def make_frame():
    rows = []
    for s in splits:
        rows.append(make_row(s, f"p_{s}", f"fp_{s}", 1, "case0"))
        rows += [make_row(s, f"e{k}_{s}", f"fe{k}_{s}", 0, "case3") for k in range(8)]
        rows.append(make_row(s, f"x_{s}", f"y_{s}", 0, "case2s"))
    return pd.DataFrame(rows)

# `pairs` stand-in = the set before §13: the same frame, with each swap row still an easy negative (same counts, same ids)
pairs_t = make_frame().replace({"neg_type": {"case2s": "case3"}})

def edit(**cols):
    def apply(frame, i):
        for col, value in cols.items():
            frame.at[i, col] = value
    return apply

def add_swap_on_top(frame, i):
    # the "add on top" bug: an extra swap row with no easy negative removed (ids known, addresses unique -> only #6 should fire)
    row = frame.loc[i].copy()
    row["address_a"], row["address_b"] = "12 Oak Street", "12 Elm Avenue"
    frame.loc[len(frame)] = row

cases = {
    "good frame":          (None,                    edit()),
    "id_a == id_b":        ("#1 id_a equals id_b",   edit(id_b="x_train")),
    "zip mismatch":        ("#1 zip_a doesn't",      edit(zip_b="06902")),
    "number mismatch":     ("#1 number_a doesn't",   edit(number_b="99")),
    "Y in another split":  ("#2",                    edit(id_b="y_val")),
    "libpostal overlap":   ("#3",                    edit(street_b="Main St")),
    "duplicate pair":      ("#4",                    edit(address_a="14 Main Street", address_b="14 Maple Avenue")),
    "B has no number":     ("#5",                    edit(address_b="Maple Avenue")),
    "no number on A or B": (None,                    edit(address_a="Main Street", address_b="Maple Avenue", number_a=None, number_b=None)),
    "split not 1:9":       ("#6",                    edit(label=1)),
    "swap added on top":   ("#6",                    add_swap_on_top),
}

for name, (expected, mutate) in cases.items():
    frame = make_frame()
    swap_row = frame.index[(frame["split"] == "train") & (frame["neg_type"] == "case2s")][0]
    mutate(frame, swap_row)
    messages = [issue[0] for issue in flag_issues(frame, pairs_t)]
    ok = not messages if expected is None else any(expected in m for m in messages)
    print("PASS" if ok else "FAIL", name, "|", [m.split(". Rows")[0] for m in messages])


PASS good frame | []
PASS id_a == id_b | ['#1 id_a equals id_b']
PASS zip mismatch | ["#1 zip_a doesn't match zip_b"]
PASS number mismatch | ["#1 number_a doesn't match number_b"]
PASS Y in another split | ["#2 id_a or id_b is not in the row's split"]
PASS libpostal overlap | ['#3 street_a and street_b share a libpostal expansion']
PASS duplicate pair | ['#4 (address_a, address_b) is duplicated']
PASS B has no number | ["#5 address_b doesn't match ^\\d+ \\S"]
PASS split not 1:9 | ['#6 train: negative share 0.800, expected 0.9']


## 12. Counts → decide amounts (D6)
**Does:** candidates per split × `swap_type`, distinct X places, rows per street pair. Then decide the per-X cap,
train share and eval size (≥ 400 across val + test).  
**Input:** `swaps`

In [ ]:
# §12: counts per split x swap_type -> input for D6


## 13. Sample, replace easy negatives, write
**Does:** apply D6 (seeded) → remove the same number of easy negatives per split (seeded) → write new pair files →
point `load_pairs` at them. Feature / libpostal caches will then raise as stale: delete and rebuild.  
**Input → output:** `pairs` + `swaps` → new pair files

In [ ]:
# §13: sample swaps, drop the same number of case3 per split, write files

# Diagnostics
One-off inspections kept out of the pipeline. Each has its own cell below; when done, fill in **Result** here.

### D0 · Why does `pairs` have 288,277 rows when 02's result note says 288,294?
**Run after:** Load  
**How:** compare per split × `neg_type` against 02's result table; count `pairs["zip_b"].isna().sum()`  
**Red flag:** the 17-row gap isn't explained by the 17 NULL `zip_b` rows  
**Result:** _pending_

In [ ]:
# D0: Why does `pairs` have 288,277 rows when 02's result note says 288,294?


### D1 · Do high-JW and low-JW street pairs look different?
**Run after:** §8, §9  
**How:** seeded samples from the top and bottom JW bands, per `swap_type`  
**Red flag:** high band = still the same street; low band = not really look-alike  
**Result:** _pending_

In [ ]:
# D1: Do high-JW and low-JW street pairs look different?

### D2 · Are the swaps about as hard as real case2?
**Run after:** §11  
**How:** JW distribution of `case2s` vs real `case2` (same clean-street JW)  
**Red flag:** `case2s` far above or below real case2  
**Result:** _pending_

In [ ]:
# D2: Are the swaps about as hard as real case2?

### D3 · Do the swapped B strings look like real Foursquare strings?
**Run after:** §11  
**How:** B length; share with a suite / unit / town: `case2s` vs `case0` B strings  
**Red flag:** a clear difference (a trace the model could learn)  
**Result:** _pending_

In [ ]:
# D3: Do the swapped B strings look like real Foursquare strings?

### D4 · Is the slice concentrated?
**Run after:** §11  
**How:** rows per X and per street pair; top 10 of each  
**Red flag:** a few streets or places supply most rows  
**Result:** _pending_

In [ ]:
# D4: Is the slice concentrated?

### D5 · Do the filters drop the right rows?
**Run after:** §3, §6, §7  
**How:** seeded sample of what each filter dropped  
**Red flag:** a dropped pair that should have stayed, or a kept one that should have gone  
**Result:** _pending_

In [ ]:
# D5: Do the filters drop the right rows?